# Puertas y construcción de circuitos

[`01_primer_circuito.ipynb`](01_primer_circuito.ipynb) usó dos puertas, H y CX, para construir el estado de Bell. Polypus ofrece un conjunto más amplio de puertas, y este notebook las recorre, junto con algunas formas más de construir circuitos.

## Puertas de un qubit

| Puerta | Qué hace |
|---|---|
| `h(q)` | Pone el qubit en superposición, como se vio en [`00b`](00b_intuicion_cuantica.ipynb) y en [`01_primer_circuito.ipynb`](01_primer_circuito.ipynb). |
| `x(q)` | Invierte 0 y 1: el NOT cuántico. |
| `y(q)` | Igual que `x`, y además cambia la fase del estado. |
| `z(q)` | Cambia la fase del estado sin tocar el valor medido en la base habitual. Su efecto solo se aprecia combinado con superposición: sobre un qubit en un valor ya definido, una medida no distingue si se aplicó o no. |
| `s(q)`, `sdg(q)` | Giros de fase de un cuarto de vuelta, en un sentido y en el otro. |
| `t(q)`, `tdg(q)` | Giros de fase de un octavo de vuelta. La T reaparece más adelante en la serie, al descomponer puertas más complejas. |
| `rx(q, theta)`, `ry(q, theta)`, `rz(q, theta)` | Rotaciones continuas alrededor de cada eje, en un ángulo `theta` elegido. |
| `u(q, theta, phi, lam)` | La puerta de un qubit más general: cualquier operación sobre un único qubit se puede expresar así. |

`x`, `y` y `z` son, en el fondo, casos concretos de las rotaciones `rx`/`ry`/`rz`, con el ángulo fijado a π: la diferencia con estas últimas es que su ángulo es fijo en vez de elegirse. `h` no entra en este grupo: no es una rotación alrededor de uno de esos tres ejes, sino una combinación de `x` y `z`.

A diferencia de `h`, que siempre reparte las probabilidades al 50%, una rotación como `rx` permite elegir la proporción:

In [ ]:
import math

import polypus

qc = polypus.Circuit(1).rx(0, math.pi / 3).measure_all()
result = polypus.run_quantum_circuit(qc, shots=1000, infrastructure="local")
print(result.counts[0])

El resultado se acerca a un 25% de `'1'` y un 75% de `'0'`, en vez del 50/50 de `h`. Cambiar el ángulo cambia esa proporción: es la base de los circuitos entrenables que aparecen en [`06a_entrenamiento_variacional.ipynb`](06a_entrenamiento_variacional.ipynb).

## Puertas de dos qubits

| Puerta | Qué hace |
|---|---|
| `cx(q0, q1)` | CNOT: invierte `q1` si `q0` vale 1, como se vio en [`01_primer_circuito.ipynb`](01_primer_circuito.ipynb). |
| `cz(q0, q1)` | Como `cx`, pero cambia la fase en vez del valor, y es simétrica: no hay distinción real entre control y objetivo. |
| `swap(q0, q1)` | Intercambia el estado de los dos qubits. |
| `rzz(q0, q1, theta)`, `rxx(q0, q1, theta)` | Rotaciones de dos qubits, con ángulo elegido. Muy usadas en los circuitos entrenables de [`06a_entrenamiento_variacional.ipynb`](06a_entrenamiento_variacional.ipynb). |
| `cp(q0, q1, theta)` | Fase controlada: como `cz`, pero con un ángulo elegido en vez de fijo. Reaparece en la transformada de Fourier cuántica, [`04b_qft.ipynb`](04b_qft.ipynb). |

## Otras operaciones

- `barrier()`: no cambia el estado del circuito, solo marca una separación visual u organizativa entre grupos de puertas.
- `measure(qubit, cbit)`: mide un único qubit concreto, a diferencia de `measure_all()`, que mide todos a la vez.

Las dos tablas de arriba son las puertas que usa el resto de la serie, no la lista completa: Polypus también tiene versiones controladas de varias de ellas (`ch`, `crx`, `cry`, `crz`, `cu`...), puertas multi-controladas como `ccx` o `c3x`, usadas en [`05b_grover.ipynb`](05b_grover.ipynb), y algunas puertas heredadas de Qiskit con su propio nombre (`p`, `u1`, `u2`, `sx`...). La lista completa está en `dir(polypus.Circuit)`.

## Un ejemplo más: el estado GHZ

El mismo patrón del estado de Bell, una H seguida de CX, se extiende de forma natural a más qubits. Con tres qubits y dos CX desde el mismo control se obtiene el llamado **estado GHZ**:

In [ ]:
ghz = polypus.Circuit(3).h(0).cx(0, 1).cx(0, 2).measure_all()
result = polypus.run_quantum_circuit(ghz, shots=1000, infrastructure="local")
print(result.counts[0])

Igual que con el estado de Bell, solo aparecen `'000'` y `'111'`, nunca una mezcla: los tres qubits quedan entrelazados a la vez.

## Visualizar un circuito

Un circuito de Polypus se dibuja solo, sin ninguna librería adicional. `print` muestra el dibujo como texto, con una línea por qubit, que se lee de izquierda a derecha:

In [ ]:
print(ghz)

H actúa sobre el qubit 0, cada CX une su control en `q0` con una X en otro qubit, y cada `M` es una medida. La línea doble de abajo guarda los bits clásicos: cada medida baja hasta ella, y el número de debajo es el bit que guarda su resultado.

Cuando un circuito está en la última línea de una celda, el notebook muestra el mismo dibujo como imagen:

In [ ]:
ghz

Esto ocurre también con una línea que añade una puerta, como `qc.measure_all()`, porque cada método de puerta devuelve el propio circuito, como se vio en [`00a`](00a_programacion_basica.ipynb): a partir de aquí, muchas celdas que construyen un circuito acaban mostrando su dibujo.

`ghz.draw()` devuelve el dibujo de texto como cadena, y `ghz.draw("svg")` devuelve la imagen como texto SVG, lista para guardarla en un fichero. El dibujo de texto se parte a los 80 caracteres por defecto; `fold` cambia ese ancho, y `fold=None` no lo parte nunca.

## Resumen

Este notebook ha recorrido el resto de puertas de Polypus, de un qubit y de dos, con dos ejemplos ejecutados: una rotación con probabilidad ajustable, y una extensión del estado de Bell a tres qubits. También ha mostrado cómo un circuito se dibuja solo, como texto o como imagen.

[`02b_interoperabilidad_qiskit_qasm.ipynb`](02b_interoperabilidad_qiskit_qasm.ipynb) cubre cómo estos circuitos interoperan con Qiskit y con OpenQASM.

## Siguiente paso

Continuar con: [`02b_interoperabilidad_qiskit_qasm.ipynb`](02b_interoperabilidad_qiskit_qasm.ipynb).